In [11]:
# -*- coding: utf-8 -*-
import sys
import os
import traceback
import h5py
import pandas as pd
import numpy as np
import gc
from tensorflow.keras import backend as K
from tqdm import tqdm
from datetime import datetime
from scipy.signal import butter, filtfilt

# 1. PAKSA KERAS LEGACY & BERSIHKAN TERMINAL
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow import keras

# Pastikan path Library Zhi Geng terdaftar
BASE_REP = '/Volumes/Extreme SSD/mcquake_ori_file/Code & Figure demo'
if BASE_REP not in sys.path:
    sys.path.append(BASE_REP)

from Library import utils, dataset

#===================================================================
# FUNGSI BANDPASS FILTER (1 Komponen)
#===================================================================
def bandpass_filter_1c(data, lowcut=5.0, highcut=20.0, fs=100.0, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

if __name__ == "__main__":
    #===================================================================
    # 1. KONFIGURASI PATH & OUTPUT
    #===================================================================
    CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
    HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
    SAVE_DIR = '/Volumes/Extreme SSD/stream_stead/output'
    
    if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)
    
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    
    # [PERBAIKAN KRUSIAL] Path embedding Z menggunakan UUSS (Sesuai origin 1C Zhi Geng)
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    #===================================================================
    # 2. LOAD MODEL & 1D PDF (MURNI ZHI GENG)
    #===================================================================
    print("[INFO] Memuat Model dan Kurva PDF 1D (Sumbu Z) UUSS...")
    try:
        embedding_model = keras.models.load_model(filepath=MODEL_PATH)
        
        # MURNI 1C: Hanya muat JSON sumbu Z dan buat PDF 1D
        embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
        embeddings_Z_PDFs = utils.embedding_PDFs_1D(embedding_Z)
    except Exception as e:
        print(f"❌ Gagal memuat Model atau JSON: {e}")
        sys.exit(1)

    #===================================================================
    # 3. LOAD METADATA STEAD
    #===================================================================
    print("[INFO] Membaca metadata STEAD (1.2M Data)...")
    df = pd.read_csv(CSV_PATH, low_memory=False)
    df = df[df['trace_category'].isin(['earthquake_local', 'noise'])]
    
    total_true, total_pred = [], []
    num_points = 700 
    checkpoint_step = 100000 

    #===================================================================
    # 4. BIG DATA STREAMING INFERENCE (SEQUENTIAL M3 PRO)
    #===================================================================
    print("\n🚀 MEMULAI EKSEKUSI BIG DATA (ESTIMASI ~4 JAM)...")
    with h5py.File(HDF5_PATH, 'r') as f:
        data_group = f['data']
        
        for i, (idx, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="STEAD 1.2M 1C")):
            try:
                trace_id = row['trace_name']
                category = row['trace_category']
                
                # Load wave [6000, 3] dari HDF5
                wave_data = data_group[trace_id][()]
                
                # Prosedur pemotongan jendela (Windowing)
                if category == 'earthquake_local':
                    p_arrival = int(row['p_arrival_sample'])
                    start_idx = p_arrival - 50
                    end_idx = start_idx + num_points
                    if end_idx > 6000 or start_idx < 0: continue
                    # Ambil sumbu Z (indeks 2)
                    z_component = wave_data[start_idx:end_idx, 2] 
                    true_label = 1
                else:
                    z_component = wave_data[:num_points, 2] 
                    true_label = 0
                
                #=======================================================
                # PRE-PROCESSING KHUSUS SUMBU Z
                #=======================================================
                # 1. Bandpass Filter [PERBAIKAN: Lowcut 5.0 Hz sesuai nama model]
                z_component = bandpass_filter_1c(z_component, lowcut=5.0, highcut=20.0, fs=100.0)
                
                # 2. Demeaning khusus sumbu Z
                z_component -= np.mean(z_component)
                
                # 3. Normalisasi Max-Absolute khusus sumbu Z
                norm_val = np.max(np.abs(z_component))
                if norm_val > 0: 
                    z_component /= norm_val
                
                # Reshape ke (1, 700, 1)
                z_input = z_component.astype(np.float32).reshape(1, -1, 1)
                
                #=======================================================
                # EKSTRAKSI & INFERENSI (MURNI ZHI GENG)
                #=======================================================
                # Ekstraksi Laten 1D (Memanggil model sebagai callable jauh lebih cepat dari .predict)
                _in_Z = embedding_model(z_input, training=False).numpy()[0].reshape(1, -1)[0]
                
                # Inferensi 1D KDE murni
                p_pred, _, _ = utils.infer_1C_PDFs(_in_Z, embeddings_Z_PDFs, "Kernel")
                
                # Mapping hasil: 0=Noise, 1=Noise/Gempa (Kita jadikan Gempa jika > 0)
                if p_pred < 2:
                    total_true.append(true_label)
                    total_pred.append(p_pred)
                elif p_pred == 2:
                    total_true.append(true_label)
                    total_pred.append(1)

                #=======================================================
                # CHECKPOINT & PEMBESIH MEMORI (ANTI-LEAK)
                #=======================================================
                if (i + 1) % checkpoint_step == 0:
                    temp_matrix, temp_metrics = utils.calc_confusion_metrics(total_true, total_pred)
                    dataset.save_json_data(os.path.join(SAVE_DIR, f"checkpoint_1c_{i+1}.json"), temp_metrics)
                
                # BERSIHKAN SAMPAH TENSOR SETIAP 1000 ITERASI
                if (i + 1) % 1000 == 0:
                    K.clear_session()
                    gc.collect()
            except Exception:
                continue

    #===================================================================
    # 5. FINAL RESULTS
    #===================================================================
    matrix, metrics = utils.calc_confusion_metrics(total_true, total_pred)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    dataset.save_json_data(os.path.join(SAVE_DIR, f"FINAL_STEAD_1.2M_1C_{timestamp}.json"), metrics)
    
    fig = utils.plot_confusion("MCU_5-20 STEAD 1.2M 1C (Z-Only)", ["NO", "LE"], matrix, metrics)
    fig.savefig(os.path.join(SAVE_DIR, f"Final_Confusion_Matrix_1C_{timestamp}.jpg"), dpi=300)
    
    print(f"\n[SUKSES] Evaluasi Big Data STEAD 1C Selesai!")
    print(f"Akurasi Akhir: {metrics.get('Accuracy (avg.)')}")

[INFO] Memuat Model dan Kurva PDF 1D (Sumbu Z) UUSS...
[INFO] Membaca metadata STEAD (1.2M Data)...

🚀 MEMULAI EKSEKUSI BIG DATA (ESTIMASI ~4 JAM)...


STEAD 1.2M 1C:   1%|          | 6771/1265657 [01:34<4:52:00, 71.85it/s] 


KeyboardInterrupt: 